# Auditoria Temporal dos Tensores Meteorologicos

Este notebook faz uma verificacao passo a passo do pipeline de dados meteorologicos usado no repositorio:

- origem dos arquivos e risco de selecao ambigua;
- alinhamento dos calendarios diarios entre precipitacao, temperatura, velocidade vertical, umidade especifica e vento;
- materializacao de tensores para um subconjunto de estacoes;
- comparacao entre o split legado (`window -> split`) e o split temporal correto (`split -> window`);
- validacao da feature sazonal anual;
- validacao da funcao `slice_intervalos_anuais` usada no notebook periodico.

Observacao pratica:

- por padrao o notebook usa apenas algumas estacoes para a etapa de materializacao dos tensores, para a execucao ficar interativa;
- para auditar todas as 62 estacoes, ajuste `STATION_LIMIT = None` na celula de configuracao.


In [1]:
from pathlib import Path
import math
import os
import sys

import numpy as np
import pandas as pd
import torch
import xarray as xr
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Execute este notebook a partir da raiz do repositorio LSTM-GNN.")

sys.path.append(str(repo_root / "src"))

from Data.feature_extraction import (
    daily_precip_dataset_to_tensor,
    daily_specific_humidity_features,
    daily_temp_features,
    daily_vertical_velocity,
    daily_wind_uv_features,
    era5_specific_humidity_tensor,
    era5_uv_to_tensor,
    forecast_steps_to_daily_precip,
    get_temp,
    get_vv,
    station_dictionary,
)
from Data.prepare_data import (
    create_sliding_windows,
    slice_intervalos_anuais,
    temporal_train_val_test_split,
)

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

WINDOW_SIZE = 30
DEMO_HORIZON = 5
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2

STATION_LIMIT = 5

TP_FILENAME = "era5_precipitation_80-26.nc"
VV_FILENAME = "vv_94-25.nc"
TEMP_FILENAME = "temp_99-25.nc"
SH_FILENAME = "era5_sh_00-25.nc"
WIND_FILENAME = "era5_wind_14-25.nc"

nc_dir = repo_root / "Datasets" / "nc_files"
catalog_file = next((repo_root / "Datasets" / "dados_inmet").glob("Catalogo*.csv"))

print({
    "repo_root": str(repo_root),
    "periodo": f"{START_DATE} -> {END_DATE}",
    "window_size": WINDOW_SIZE,
    "demo_horizon": DEMO_HORIZON,
    "station_limit": STATION_LIMIT,
})


{'repo_root': 'c:\\Local Reposity\\LSTM-GNN', 'periodo': '2015-01-01 -> 2025-12-31', 'window_size': 30, 'demo_horizon': 5, 'station_limit': 5}


In [2]:
file_audit = []
for variable in ["precipitation", "vv", "temp", "sh", "wind"]:
    matches = sorted([p.name for p in nc_dir.glob(f"*{variable}*")])
    file_audit.append({
        "variable": variable,
        "n_matches": len(matches),
        "matches": matches,
        "smart_load_dataset_is_safe": len(matches) == 1,
    })

display(pd.DataFrame(file_audit))
print(
    "Observacao: smart_load_dataset() abre o primeiro match do diretorio. "
    "Para 'temp' existem dois arquivos, entao o carregamento generico nao e deterministicamente seguro para auditorias completas."
)

tp_path = nc_dir / TP_FILENAME
vv_path = nc_dir / VV_FILENAME
temp_path = nc_dir / TEMP_FILENAME
sh_path = nc_dir / SH_FILENAME
wind_path = nc_dir / WIND_FILENAME

rea_tp = xr.open_dataset(tp_path).sel(time=slice(START_DATE, END_DATE))
rea_vv = xr.open_dataset(vv_path).sel(time=slice(START_DATE, END_DATE))
rea_temp = xr.open_dataset(temp_path).sel(time=slice(START_DATE, END_DATE))
rea_sh = xr.open_dataset(sh_path).sel(time=slice(START_DATE, END_DATE))
rea_wind = xr.open_dataset(wind_path).sel(time=slice(START_DATE, END_DATE))

tp_daily = forecast_steps_to_daily_precip(rea_tp, to_mm=True)
vv_daily = daily_vertical_velocity(rea_vv, var_name="w", load_into_memory=False)
temp_daily = daily_temp_features(rea_temp)
sh_daily = daily_specific_humidity_features(rea_sh, var_name="q")
wind_daily = daily_wind_uv_features(rea_wind)

print("Arquivos explicitamente usados nesta auditoria:")
print({
    "tp": tp_path.name,
    "vv": vv_path.name,
    "temp": temp_path.name,
    "sh": sh_path.name,
    "wind": wind_path.name,
})


,variable,n_matches,matches,smart_load_dataset_is_safe
0,precipitation,1,[era5_precipitation_80-26.nc],True
1,vv,1,[vv_94-25.nc],True
2,temp,2,"[temp_05-25.nc, temp_99-25.nc]",False
3,sh,1,[era5_sh_00-25.nc],True
4,wind,1,[era5_wind_14-25.nc],True


Observacao: smart_load_dataset() abre o primeiro match do diretorio. Para 'temp' existem dois arquivos, entao o carregamento generico nao e deterministicamente seguro para auditorias completas.
Arquivos explicitamente usados nesta auditoria:
{'tp': 'era5_precipitation_80-26.nc', 'vv': 'vv_94-25.nc', 'temp': 'temp_99-25.nc', 'sh': 'era5_sh_00-25.nc', 'wind': 'era5_wind_14-25.nc'}


In [3]:
def summarize_index(name, idx):
    return {
        "serie": name,
        "start": idx[0],
        "end": idx[-1],
        "len": len(idx),
    }


def compare_indexes(name_a, idx_a, name_b, idx_b):
    return {
        "comparacao": f"{name_a} vs {name_b}",
        "equal": idx_a.equals(idx_b),
        "len_a": len(idx_a),
        "len_b": len(idx_b),
        "intersection": len(idx_a.intersection(idx_b)),
        "only_a_head": [str(x.date()) for x in idx_a.difference(idx_b)[:3]],
        "only_b_head": [str(x.date()) for x in idx_b.difference(idx_a)[:3]],
    }


tp_idx = pd.DatetimeIndex(pd.to_datetime(tp_daily.time.values))
tp_idx_drop_last = pd.DatetimeIndex(pd.to_datetime(tp_daily.isel(time=slice(None, -1)).time.values))
temp_idx = pd.DatetimeIndex(pd.to_datetime(temp_daily.time.values))
vv_idx = pd.DatetimeIndex(pd.to_datetime(vv_daily.time.values))
sh_idx = pd.DatetimeIndex(pd.to_datetime(sh_daily.time.values))
wind_idx = pd.DatetimeIndex(pd.to_datetime(wind_daily.time.values))
wind_idx_drop_last = pd.DatetimeIndex(pd.to_datetime(wind_daily.isel(time=slice(None, -1)).time.values))

display(pd.DataFrame([
    summarize_index("tp_daily", tp_idx),
    summarize_index("tp_daily[:-1]", tp_idx_drop_last),
    summarize_index("temp_daily", temp_idx),
    summarize_index("vv_daily", vv_idx),
    summarize_index("sh_daily", sh_idx),
    summarize_index("wind_daily", wind_idx),
]))

display(pd.DataFrame([
    compare_indexes("tp_daily[:-1]", tp_idx_drop_last, "temp_daily", temp_idx),
    compare_indexes("tp_daily[:-1]", tp_idx_drop_last, "vv_daily", vv_idx),
    compare_indexes("tp_daily[:-1]", tp_idx_drop_last, "sh_daily", sh_idx),
    compare_indexes("tp_daily[:-1]", tp_idx_drop_last, "wind_daily", wind_idx),
    compare_indexes("wind_daily[:-1]", wind_idx_drop_last, "temp_daily", temp_idx),
]))

assert tp_idx[-1] == pd.Timestamp("2026-01-01")
assert tp_idx_drop_last.equals(temp_idx)
assert tp_idx_drop_last.equals(vv_idx)
assert tp_idx_drop_last.equals(sh_idx)
assert tp_idx_drop_last.equals(wind_idx)
assert not wind_idx_drop_last.equals(temp_idx)

print(
    "Validacao principal: a precipitacao precisa remover apenas o ultimo dia extra (2026-01-01). "
    "Temperatura, vv, sh e vento ja estao alinhados sem esse corte adicional."
)


,serie,start,end,len
0,tp_daily,2015-01-01,2026-01-01,4019
1,tp_daily[:-1],2015-01-01,2025-12-31,4018
2,temp_daily,2015-01-01,2025-12-31,4018
3,vv_daily,2015-01-01,2025-12-31,4018
4,sh_daily,2015-01-01,2025-12-31,4018
5,wind_daily,2015-01-01,2025-12-31,4018


,comparacao,equal,len_a,len_b,intersection,only_a_head,only_b_head
0,tp_daily[:-1] vs temp_daily,True,4018,4018,4018,[],[]
1,tp_daily[:-1] vs vv_daily,True,4018,4018,4018,[],[]
2,tp_daily[:-1] vs sh_daily,True,4018,4018,4018,[],[]
3,tp_daily[:-1] vs wind_daily,True,4018,4018,4018,[],[]
4,wind_daily[:-1] vs temp_daily,False,4017,4018,4017,[],[2025-12-31]


Validacao principal: a precipitacao precisa remover apenas o ultimo dia extra (2026-01-01). Temperatura, vv, sh e vento ja estao alinhados sem esse corte adicional.


In [4]:
catalog = pd.read_csv(catalog_file, sep=";")
stations_full = station_dictionary(catalog)
if STATION_LIMIT is None:
    stations = stations_full
else:
    stations = dict(list(stations_full.items())[:STATION_LIMIT])

T_expected = (pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days + 1
N_selected = len(stations)

X_tp = daily_precip_dataset_to_tensor(tp_daily.isel(time=slice(None, -1)), stations).unsqueeze(-1)
X_temp = get_temp(START_DATE, END_DATE, temp_daily, stations)
X_vv = get_vv(START_DATE, END_DATE, vv_daily, stations)
X_sh = era5_specific_humidity_tensor(str(sh_path), stations, start=START_DATE, end=END_DATE, var_name="q")
X_wind_raw = era5_uv_to_tensor(str(wind_path), stations, start=START_DATE, end=END_DATE, u_var="u", v_var="v")
X_wind_pipeline = X_wind_raw.reshape(T_expected, N_selected, -1)

tensor_summary = pd.DataFrame([
    {"tensor": "X_tp", "shape": tuple(X_tp.shape), "days_ok": X_tp.shape[0] == T_expected},
    {"tensor": "X_temp", "shape": tuple(X_temp.shape), "days_ok": X_temp.shape[0] == T_expected},
    {"tensor": "X_vv", "shape": tuple(X_vv.shape), "days_ok": X_vv.shape[0] == T_expected},
    {"tensor": "X_sh", "shape": tuple(X_sh.shape), "days_ok": X_sh.shape[0] == T_expected},
    {"tensor": "X_wind_raw", "shape": tuple(X_wind_raw.shape), "days_ok": X_wind_raw.shape[0] == T_expected},
    {"tensor": "X_wind_pipeline", "shape": tuple(X_wind_pipeline.shape), "days_ok": X_wind_pipeline.shape[0] == T_expected},
])

display(tensor_summary)

assert tensor_summary["days_ok"].all()

print(
    f"Materializacao validada para {N_selected} estacoes. "
    f"Para auditar todas as estacoes, ajuste STATION_LIMIT = None e reexecute esta celula."
)


,tensor,shape,days_ok
0,X_tp,"(4018, 5, 1)",True
1,X_temp,"(4018, 5, 6)",True
2,X_vv,"(4018, 5, 12)",True
3,X_sh,"(4018, 5, 6)",True
4,X_wind_raw,"(4018, 5, 2, 2)",True
5,X_wind_pipeline,"(4018, 5, 4)",True


Materializacao validada para 5 estacoes. Para auditar todas as estacoes, ajuste STATION_LIMIT = None e reexecute esta celula.


In [5]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")
X_dummy = torch.arange(len(dates), dtype=torch.float32).reshape(len(dates), 1, 1)
y_dummy = torch.arange(len(dates), dtype=torch.float32).reshape(len(dates), 1)

Xs_old, ys_old = create_sliding_windows(
    X_dummy,
    y_dummy,
    window_size=WINDOW_SIZE,
    horizon=DEMO_HORIZON,
)

B = Xs_old.shape[0]
n_train = int(TRAIN_RATIO * B)
n_val = int(VAL_RATIO * B)

old_train_last = [dates[int(v)] for v in ys_old[n_train - 1, :, 0].tolist()]
old_val_first = [dates[int(v)] for v in ys_old[n_train, :, 0].tolist()]
old_val_last = [dates[int(v)] for v in ys_old[n_train + n_val - 1, :, 0].tolist()]
old_test_first = [dates[int(v)] for v in ys_old[n_train + n_val, :, 0].tolist()]

X_train_new, y_train_new, X_val_new, y_val_new, X_test_new, y_test_new = temporal_train_val_test_split(
    X_dummy,
    y_dummy,
    window_size=WINDOW_SIZE,
    horizon=DEMO_HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    use_context=True,
)

new_train_last = [dates[int(v)] for v in y_train_new[-1, :, 0].tolist()]
new_val_first = [dates[int(v)] for v in y_val_new[0, :, 0].tolist()]
new_val_last = [dates[int(v)] for v in y_val_new[-1, :, 0].tolist()]
new_test_first = [dates[int(v)] for v in y_test_new[0, :, 0].tolist()]

split_audit = pd.DataFrame([
    {
        "pipeline": "legacy_window_then_split",
        "boundary": "train -> val",
        "left_range": f"{old_train_last[0].date()} -> {old_train_last[-1].date()}",
        "right_range": f"{old_val_first[0].date()} -> {old_val_first[-1].date()}",
        "overlap_target_days": len(set(old_train_last).intersection(old_val_first)),
    },
    {
        "pipeline": "legacy_window_then_split",
        "boundary": "val -> test",
        "left_range": f"{old_val_last[0].date()} -> {old_val_last[-1].date()}",
        "right_range": f"{old_test_first[0].date()} -> {old_test_first[-1].date()}",
        "overlap_target_days": len(set(old_val_last).intersection(old_test_first)),
    },
    {
        "pipeline": "temporal_split_then_window",
        "boundary": "train -> val",
        "left_range": f"{new_train_last[0].date()} -> {new_train_last[-1].date()}",
        "right_range": f"{new_val_first[0].date()} -> {new_val_first[-1].date()}",
        "overlap_target_days": len(set(new_train_last).intersection(new_val_first)),
    },
    {
        "pipeline": "temporal_split_then_window",
        "boundary": "val -> test",
        "left_range": f"{new_val_last[0].date()} -> {new_val_last[-1].date()}",
        "right_range": f"{new_test_first[0].date()} -> {new_test_first[-1].date()}",
        "overlap_target_days": len(set(new_val_last).intersection(new_test_first)),
    },
])

display(split_audit)

assert split_audit.loc[
    split_audit["pipeline"] == "legacy_window_then_split", "overlap_target_days"
].max() > 0
assert split_audit.loc[
    split_audit["pipeline"] == "temporal_split_then_window", "overlap_target_days"
].max() == 0

print(
    "Conclusao: para horizontes maiores que 1, o fluxo legado (janela antes do split) "
    "mistura dias-alvo entre treino/validacao/teste. O split temporal correto elimina essa sobreposicao."
)


,pipeline,boundary,left_range,right_range,overlap_target_days
0,legacy_window_then_split,train -> val,2022-09-18 -> 2022-09-22,2022-09-19 -> 2022-09-23,4
1,legacy_window_then_split,val -> test,2024-11-22 -> 2024-11-26,2024-11-23 -> 2024-11-27,4
2,temporal_split_then_window,train -> val,2022-09-08 -> 2022-09-12,2022-09-13 -> 2022-09-17,0
3,temporal_split_then_window,val -> test,2024-11-19 -> 2024-11-23,2024-11-24 -> 2024-11-28,0


Conclusao: para horizontes maiores que 1, o fluxo legado (janela antes do split) mistura dias-alvo entre treino/validacao/teste. O split temporal correto elimina essa sobreposicao.


In [ ]:
real_dates = pd.date_range(START_DATE, END_DATE, freq="D")
calendar_notebook = torch.tensor(
    [math.sin(math.pi * (t % 365 + 1) / 365) for t in range(T_expected)],
    dtype=torch.float32,
)
calendar_real_sin = torch.tensor(
    [
        math.sin(
            2 * math.pi * (d.timetuple().tm_yday - 1) / (366 if d.is_leap_year else 365)
        )
        for d in real_dates
    ],
    dtype=torch.float32,
)
calendar_real_cos = torch.tensor(
    [
        math.cos(
            2 * math.pi * (d.timetuple().tm_yday - 1) / (366 if d.is_leap_year else 365)
        )
        for d in real_dates
    ],
    dtype=torch.float32,
)

calendar_diff = (calendar_notebook - calendar_real_sin).abs()
sample_idx = [i for i in [0, 58, 59, 365, 366, 424, 425] if i < T_expected]

calendar_audit = pd.DataFrame({
    "date": [str(real_dates[i].date()) for i in sample_idx],
    "notebook_feature": [float(calendar_notebook[i]) for i in sample_idx],
    "real_calendar_sin": [float(calendar_real_sin[i]) for i in sample_idx],
    "abs_diff": [float(calendar_diff[i]) for i in sample_idx],
})

display(calendar_audit)
print({
    "max_abs_diff": float(calendar_diff.max()),
    "mean_abs_diff": float(calendar_diff.mean()),
})

X_year_recommended = torch.tensor(
    np.stack([calendar_real_sin.numpy(), calendar_real_cos.numpy()], axis=-1),
    dtype=torch.float32,
).unsqueeze(1).repeat(1, N_selected, 1)

print("Shape sugerido para a feature sazonal corrigida:", tuple(X_year_recommended.shape))
print(
    "Conclusao: a feature atual do notebook nao usa a data real do calendario e ainda usa meia onda (pi) em vez de ciclo completo (2*pi)."
)


In [ ]:
toy_time = pd.date_range("2020-01-01", "2025-12-31", freq="D")
toy = xr.Dataset({"flag": ("time", np.arange(len(toy_time), dtype=np.int32))}, coords={"time": toy_time})
toy_out = slice_intervalos_anuais(
    toy,
    np.datetime64("2020-01-01"),
    np.datetime64("2025-12-31"),
    months=2,
    days=10,
)

toy_idx = pd.DatetimeIndex(pd.to_datetime(toy_out.time.values))
toy_summary = []
for year in sorted(toy_idx.year.unique()):
    sub = toy_idx[toy_idx.year == year]
    toy_summary.append({
        "year": int(year),
        "n_days": len(sub),
        "start": sub.min(),
        "end": sub.max(),
    })

display(pd.DataFrame(toy_summary))
print(
    "Se months=2 e days=10, o esperado seria aproximadamente 71 dias por ano. "
    "A funcao atual retorna ~32 dias porque fixa months=1 internamente."
)


## Checklist de interpretacao

Se este notebook rodar como esperado, as conclusoes principais devem ser:

1. `forecast_steps_to_daily_precip()` gera um dia extra no fim por causa do ultimo `valid_time`; para precipitacao, cortar apenas o ultimo dia alinha o calendario com as demais variaveis.
2. Nao se deve aplicar o mesmo `[:-1]` ao vento horario agregado, porque isso remove um dia real (`2025-12-31`) e cria desalinhamento.
3. O fluxo legado `create_sliding_windows(...) -> train_split(...)` nao e seguro para `horizon > 1`, porque os dias-alvo ficam sobrepostos entre splits.
4. A feature sazonal anual dos notebooks atuais nao representa corretamente o dia real do calendario.
5. `slice_intervalos_anuais()` precisa ser corrigida antes de uso generico.
6. Quando houver mais de um arquivo por variavel, o carregamento por substring precisa ser travado em arquivo explicito para evitar cobertura temporal ambigua.
